In [5]:
# ==============================================================
# Sales Forecast v4
# Exog methods:
#  - Prophet, SARIMA, ETS (single)
#  - Ensemble(Prophet+SARIMA+ETS)  [stat-ensemble]
#  - ML-Exog RF, ML-Exog XGB (recursive ML)
#  - ML-Exog RF+XGB (exog ensemble via VAL inverse-MAE)
#  - All-Ensemble-5 (Prophet+SARIMA+ETS+MLRF+MLXGB)
#  - All-Ensemble-Top2 / Top3 (by VAL Y-ENS MAE)
#
# Y variants: RF, XGB, Y-ENS (RF+XGB)
# Adds: ROCV tuning, Bootstrap PI (80/95), SHAP/Importances, Stockout P(3m/6m)
# Prints all major tables to console AND saves to outputs/
# ============================================================== 

import warnings, logging, os, sys
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
#warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="Maximum Likelihood optimization failed to converge")
warnings.filterwarnings("ignore", message="Optimization failed to converge")
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("statsmodels").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product
from collections import OrderedDict

from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Optional SHAP
try:
    import shap
    HAVE_SHAP = True
except Exception:
    HAVE_SHAP = False

from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ------------------ Config ------------------
CSV_PATH   = "veri_matrisi_final_sales_orders_stock_calendar_lags_fx.csv"

VAL_START  = pd.Timestamp("2024-07-01")
VAL_END    = pd.Timestamp("2024-12-01")
TEST_START = pd.Timestamp("2025-01-01")
TEST_END   = pd.Timestamp("2025-07-01")
TEST_END_SHORT = pd.Timestamp("2025-03-01")

RANDOM_STATE = 42
EXOG_VAL_H   = 6
EPS_PROPHET  = 0.05
B_BOOT       = 800
SEED         = 1337

FEATURES_Y = [
    "orders","stock",
    "orders_lag1","orders_lag3",
    "stock_lag1","stock_lag3",
    "y_lag1",
    "orders_ratio",
    "month","year",
]

STARTING_STOCK_OVERRIDE = None

# ------------------ Utils ------------------
rng_boot = np.random.default_rng(SEED)

def mae_rmse_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(np.mean((y_pred - y_true)**2))
    denom = np.where(y_true == 0, 1, y_true)
    mape = np.mean(np.abs((y_true - y_pred) / denom)) * 100
    return mae, rmse, mape

def ensure_ms_freq(df):
    d = df.copy().sort_values("ds")
    d["ds"] = d["ds"].dt.to_period("M").dt.to_timestamp(how="start")
    d = d.set_index("ds").sort_index()
    d.index = pd.DatetimeIndex(d.index, freq="MS")
    return d.reset_index()

def add_calendar(df):
    d = df.copy()
    d["year"]  = d["ds"].dt.year
    d["month"] = d["ds"].dt.month
    return d

def rolling_impute(s, causal=False):
    x = pd.to_numeric(s, errors="coerce")
    if causal:
        x = x.ffill()
        x = x.rolling(window=3, min_periods=1).mean()
        x = x.bfill()
    else:
        roll = x.rolling(window=3, center=True, min_periods=1).mean()
        x = x.where(~x.isna(), roll).ffill().bfill()
    return x

def smooth_causal_ma(s, window=3):
    x = pd.to_numeric(s, errors="coerce").ffill()
    return x.rolling(window=window, min_periods=1).mean().bfill()

def winsorize_series(s, lower_q=0.05, upper_q=0.95):
    x = pd.to_numeric(s, errors="coerce")
    lo = np.nanpercentile(x, lower_q*100)
    hi = np.nanpercentile(x, upper_q*100)
    return x.clip(lo, hi)

def nonneg(s):
    return pd.to_numeric(s, errors="coerce").clip(lower=0.0)

def build_lags_y(df):
    d = df.copy()
    if "orders" in d.columns and "stock" in d.columns:
        d["orders_ratio"] = d["orders"] / d["stock"].replace(0, np.nan)
    if "y" in d.columns:
        d["y_lag1"] = d["y"].shift(1)
    if "orders" in d.columns:
        d["orders_lag1"] = d["orders"].shift(1)
        d["orders_lag3"] = d["orders"].shift(3)
    if "stock" in d.columns:
        d["stock_lag1"] = d["stock"].shift(1)
        d["stock_lag3"] = d["stock"].shift(3)
    return d

def prep_features_y(df_in, causal=False):
    d = add_calendar(df_in)
    d = build_lags_y(d)
    for col in ["orders","stock"]:
        if col in d.columns:
            d[col] = rolling_impute(d[col], causal=causal)
    for col in ["orders_lag1","orders_lag3","stock_lag1","stock_lag3","y_lag1","orders_ratio"]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce").ffill().bfill().fillna(0.0)
    for c in FEATURES_Y:
        if c not in d.columns:
            d[c] = 0.0
    return d.replace([np.inf, -np.inf], np.nan).fillna(0)

# ------------------ Univariate exog models ------------------
def fit_prophet(train_df, value_col):
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False)
    m.fit(train_df.rename(columns={value_col:"y"}))
    return m

def forecast_prophet(model, steps):
    fut = model.make_future_dataframe(periods=steps, freq="MS")
    fc  = model.predict(fut)[["ds","yhat"]].tail(steps)
    return fc.rename(columns={"yhat":"yhat"})

def sarima_fit_best(y, p_range=(0,3), q_range=(0,3), P_range=(0,1), Q_range=(0,1)):
    best, best_aic = None, np.inf
    for p in range(p_range[0], p_range[1]+1):
        for q in range(q_range[0], q_range[1]+1):
            for P in range(P_range[0], P_range[1]+1):
                for Q in range(Q_range[0], Q_range[1]+1):
                    try:
                        r = SARIMAX(y, order=(p,1,q), seasonal_order=(P,1,Q,12),
                                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
                        if r.aic < best_aic:
                            best_aic = r.aic
                            best = ((p,1,q),(P,1,Q,12))
                    except Exception:
                        pass
    return best

def fit_sarima(train_df, value_col):
    y = train_df.set_index("ds")[value_col]
    y.index.freq = "MS"
    best = sarima_fit_best(y)
    if best is None:
        return None
    return SARIMAX(y, order=best[0], seasonal_order=best[1],
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

def forecast_sarima(model, steps, future_idx):
    pred = model.get_forecast(steps=steps).predicted_mean
    return pd.DataFrame({"ds": pd.DatetimeIndex(future_idx), "yhat": pred.values})

def fit_ets(train_df, value_col):
    y = train_df.set_index("ds")[value_col]
    y.index.freq = "MS"
    best_model, best_aic = None, np.inf
    for trend in ["add", "mul", None]:
        for seasonal in ["add", "mul", None]:
            try:
                for damped in [True, False]:
                    if seasonal is None:
                        model = ExponentialSmoothing(y, trend=trend, seasonal=None, damped_trend=damped).fit(optimized=True)
                    else:
                        model = ExponentialSmoothing(y, trend=trend, seasonal=seasonal, seasonal_periods=12,
                                                     damped_trend=damped).fit(optimized=True)
                    aic = getattr(model, "aic", np.inf)
                    if aic < best_aic:
                        best_aic, best_model = aic, model
            except Exception:
                continue
    if best_model is None:
        best_model = ExponentialSmoothing(y, trend="add", seasonal="add", seasonal_periods=12).fit(optimized=True)
    return best_model

def forecast_ets(model, steps, future_idx):
    pred = model.forecast(steps)
    return pd.DataFrame({"ds": pd.DatetimeIndex(future_idx), "yhat": pred.values})

def backtest_mae(series_df, value_col, method, cutoff, val_h=EXOG_VAL_H):
    s_full = series_df[["ds", value_col]].dropna().sort_values("ds")
    s = s_full[s_full["ds"] < cutoff]
    if len(s) < val_h + 6:
        return np.inf
    cut = s["ds"].max() - pd.DateOffset(months=val_h-1)
    train = s[s["ds"] < cut]
    val   = s[s["ds"] >= cut]
    steps = len(val)
    try:
        if method == "prophet":
            m = fit_prophet(train, value_col)
            yhat = forecast_prophet(m, steps)["yhat"].values
        elif method == "sarima":
            m = fit_sarima(train, value_col)
            if m is None: return np.inf
            yhat = forecast_sarima(m, steps, val["ds"].values)["yhat"].values
        else:
            m = fit_ets(train, value_col)
            yhat = forecast_ets(m, steps, val["ds"].values)["yhat"].values
    except Exception:
        return np.inf
    return mae_rmse_mape(val[value_col].values, yhat)[0]

def _postprocess_exog(df):
    d = df.copy()
    for c in ["orders","stock"]:
        if c in d.columns:
            d[c] = smooth_causal_ma(d[c], window=3)
            d[c] = winsorize_series(d[c], 0.05, 0.95)
            d[c] = nonneg(d[c])
    return d

def build_exog_univariate_for_period(df_all, start_ds, end_ds, cutoff, uni_method):
    """uni_method in {'prophet','sarima','ets'}"""
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    out = pd.DataFrame({"ds": future_idx})
    for col in ["orders","stock"]:
        s = df_all[["ds", col]].dropna().sort_values("ds")
        s = s[s["ds"] < cutoff]
        if s.empty:
            out[col] = 0.0; continue
        steps = len(future_idx)
        try:
            if uni_method == "prophet":
                m = fit_prophet(s, col);  fc = forecast_prophet(m, steps)
            elif uni_method == "sarima":
                m = fit_sarima(s, col);   fc = forecast_sarima(m, steps, future_idx)
            else:
                m = fit_ets(s, col);      fc = forecast_ets(m, steps, future_idx)
            tmp = fc.rename(columns={"yhat":col})
        except Exception:
            tmp = pd.DataFrame({"ds": future_idx, col: np.nan})
        out = out.merge(tmp[["ds",col]], on="ds", how="left")
    return _postprocess_exog(out)

def build_exog_inverse_mae_for_period(df_all, start_ds, end_ds, cutoff, eps_prophet=EPS_PROPHET):
    """Stat ensemble of Prophet+SARIMA+ETS (per var) via inverse-MAE on pre-cutoff."""
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    out = pd.DataFrame({"ds": future_idx})
    for col in ["orders","stock"]:
        s = df_all[["ds", col]].dropna().sort_values("ds")
        s = s[s["ds"] < cutoff]
        if s.empty:
            out[col] = 0.0; continue
        steps = len(future_idx)
        mae_p = backtest_mae(s, col, "prophet", cutoff)
        mae_s = backtest_mae(s, col, "sarima",  cutoff)
        mae_e = backtest_mae(s, col, "ets",     cutoff)
        try: mp  = fit_prophet(s, col); fcp = forecast_prophet(mp, steps).rename(columns={"yhat":"p"})
        except Exception: fcp = pd.DataFrame({"ds": future_idx, "p": np.nan})
        try: ms  = fit_sarima(s, col);  fcs = forecast_sarima(ms, steps, future_idx).rename(columns={"yhat":"s"})
        except Exception: fcs = pd.DataFrame({"ds": future_idx, "s": np.nan})
        try: me  = fit_ets(s, col);     fce = forecast_ets(me, steps, future_idx).rename(columns={"yhat":"e"})
        except Exception: fce = pd.DataFrame({"ds": future_idx, "e": np.nan})

        maes = np.array([mae_p, mae_s, mae_e], dtype=float)
        maes = np.where(~np.isfinite(maes) | (maes <= 0), np.nan, maes)
        inv  = 1.0 / np.where(np.isnan(maes), np.inf, maes)
        if np.isfinite(inv).sum() == 0:
            wp, ws, we = 0.60, 0.25, 0.15
        else:
            w = inv / inv.sum()
            w[0] += eps_prophet
            w = w / w.sum()
            wp, ws, we = w[0], w[1], w[2]

        tmp = (pd.DataFrame({"ds": future_idx})
                 .merge(fcp, on="ds", how="left")
                 .merge(fcs, on="ds", how="left")
                 .merge(fce, on="ds", how="left"))
        tmp[col] = (wp*tmp["p"] + ws*tmp["s"] + we*tmp["e"])
        out = out.merge(tmp[["ds",col]], on="ds", how="left")
    return _postprocess_exog(out)

# ------------------ ML-Exog (orders/stock) ------------------
EXOG_FEATURES = ["lag1","lag3","month","year"]

def make_exog_feature_frame(df_var, var_col):
    d = df_var[["ds", var_col]].sort_values("ds").copy()
    d["lag1"] = d[var_col].shift(1)
    d["lag3"] = d[var_col].shift(3)
    d["month"] = d["ds"].dt.month
    d["year"]  = d["ds"].dt.year
    return d

def train_exog_rf(df_var, var_col, cutoff):
    d = make_exog_feature_frame(df_var[df_var["ds"] < cutoff], var_col).dropna()
    if d.empty: return None
    X = d[EXOG_FEATURES]; y = d[var_col]
    rf = RandomForestRegressor(n_estimators=400, max_depth=8, min_samples_split=2, min_samples_leaf=1,
                               random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X, y); return rf

def train_exog_xgb(df_var, var_col, cutoff):
    d = make_exog_feature_frame(df_var[df_var["ds"] < cutoff], var_col).dropna()
    if d.empty: return None
    X = d[EXOG_FEATURES].to_numpy(); y = d[var_col].to_numpy()
    xgb = XGBRegressor(n_estimators=500, learning_rate=0.08, max_depth=3,
                       subsample=0.9, colsample_bytree=0.9, reg_lambda=1.2,
                       random_state=RANDOM_STATE)
    xgb.fit(X, y, verbose=False); return xgb

def recursive_forecast_exog(model, hist_df, var_col, start_ds, end_ds):
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    full = hist_df[["ds", var_col]].sort_values("ds").copy()
    out = []
    for ds in future_idx:
        tmp = make_exog_feature_frame(full, var_col)
        row = tmp[tmp["ds"] == ds][EXOG_FEATURES]
        if row.empty:
            row = pd.DataFrame({
                "lag1": [full[var_col].iloc[-1] if len(full) else 0.0],
                "lag3": [full[var_col].iloc[-3] if len(full) >= 3 else (full[var_col].iloc[-1] if len(full) else 0.0)],
                "month":[ds.month], "year":[ds.year]
            })
        x = row.to_numpy()
        yhat = float(model.predict(x)[0])
        out.append({"ds": ds, var_col: max(0.0, yhat)})
        full = pd.concat([full, pd.DataFrame({"ds":[ds], var_col:[yhat]})], ignore_index=True)
    return pd.DataFrame(out)

def build_exog_ml_for_period(df_all, start_ds, end_ds, cutoff, learner="rf"):
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    out = pd.DataFrame({"ds": future_idx})
    for col in ["orders","stock"]:
        s_hist = df_all[df_all["ds"] < cutoff][["ds", col]].copy()
        if s_hist.empty:
            out[col] = 0.0; continue
        mdl = train_exog_rf(df_all[["ds", col]].copy(), col, cutoff) if learner=="rf" \
              else train_exog_xgb(df_all[["ds", col]].copy(), col, cutoff)
        if mdl is None:
            out[col] = 0.0; continue
        fc = recursive_forecast_exog(mdl, s_hist, col, start_ds, end_ds)
        out = out.merge(fc, on="ds", how="left")
    return _postprocess_exog(out)

# ------------------ ROCV on Y ------------------
def rolling_origin_splits(df, n_splits=3, min_train_months=24):
    d = df.sort_values("ds").copy()
    if d["ds"].nunique() < (min_train_months + n_splits):
        yield (d[d["ds"] < d["ds"].max() - pd.DateOffset(months=3)],
               d[d["ds"] >= d["ds"].max() - pd.DateOffset(months=3)])
        return
    for k in range(n_splits, 0, -1):
        val_end = d["ds"].max() - pd.DateOffset(months=k-1)
        val_start = val_end - pd.DateOffset(months=2)
        train_end = val_start - pd.DateOffset(days=1)
        train = d[(d["ds"] <= train_end)]
        val   = d[(d["ds"] >= val_start) & (d["ds"] <= val_end)]
        if len(train) >= min_train_months and len(val) >= 2:
            yield train, val

def optimize_rf_rocv(df_trainval):
    base_grid = {
        "n_estimators": [400, 700],
        "max_depth": [None, 8, 12],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
    }
    best, best_score = None, np.inf
    for params in (dict(zip(base_grid.keys(), v)) for v in product(*base_grid.values())):
        maes = []
        for tr, va in rolling_origin_splits(df_trainval, n_splits=3, min_train_months=24):
            mdl = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
            mdl.fit(tr[FEATURES_Y], tr["y"])
            pred = mdl.predict(va[FEATURES_Y])
            maes.append(mean_absolute_error(va["y"], pred))
        score = np.mean(maes) if maes else np.inf
        if score < best_score:
            best_score, best = score, params
    final = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best).fit(df_trainval[FEATURES_Y], df_trainval["y"])
    return final, best, best_score

def optimize_xgb_rocv(df_trainval):
    base_grid = {
        "n_estimators": [500, 800],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 4],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "reg_lambda": [1.0, 2.0],
    }
    best, best_score = None, np.inf
    for params in (dict(zip(base_grid.keys(), v)) for v in product(*base_grid.values())):
        maes = []
        for tr, va in rolling_origin_splits(df_trainval, n_splits=3, min_train_months=24):
            mdl = XGBRegressor(random_state=RANDOM_STATE, **params)
            mdl.fit(tr[FEATURES_Y].to_numpy(), tr["y"].to_numpy(), verbose=False)
            pred = mdl.predict(va[FEATURES_Y].to_numpy())
            maes.append(mean_absolute_error(va["y"], pred))
        score = np.mean(maes) if maes else np.inf
        if score < best_score:
            best_score, best = score, params
    final = XGBRegressor(random_state=RANDOM_STATE, **best)
    final.fit(df_trainval[FEATURES_Y].to_numpy(), df_trainval["y"].to_numpy(), verbose=False)
    return final, best, best_score

# ------------------ Y recursive forward ------------------
def recursive_forward_predict_y(model, x_cols, hist_df, future_exog, start_ds, end_ds):
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    future_part = pd.DataFrame({"ds": future_idx}).merge(future_exog, on="ds", how="left")
    full = pd.concat([hist_df, future_part], ignore_index=True).sort_values("ds")

    preds = []
    for ds in future_idx:
        tmp = prep_features_y(full.copy(), causal=True)
        row = tmp.loc[tmp["ds"] == ds].copy()
        X = row[x_cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()
        y_hat = float(model.predict(X)[0])
        preds.append({"ds": ds, "yhat": y_hat})
        full.loc[full["ds"] == ds, "y"] = y_hat
        for c in ["orders","stock"]:
            if c in full.columns:
                full[c] = rolling_impute(full[c], causal=True)
    preds_df = pd.DataFrame(preds)
    used_future = full.loc[full["ds"].isin(future_idx)].copy()
    return preds_df, used_future

# ------------------ Load & prepare Y-data ------------------
os.makedirs("outputs", exist_ok=True)
os.makedirs("logs", exist_ok=True)

df_raw = pd.read_csv(CSV_PATH, parse_dates=["ds"]).sort_values("ds").reset_index(drop=True)
for c in ["y","orders","stock"]:
    if c in df_raw.columns: df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")
df_raw = ensure_ms_freq(df_raw)

mask_train = (df_raw["ds"] < VAL_START)
mask_val   = (df_raw["ds"] >= VAL_START) & (df_raw["ds"] <= VAL_END)

train_df = prep_features_y(df_raw.loc[mask_train].copy(), causal=False)
val_df   = prep_features_y(df_raw.loc[mask_val].copy(),   causal=True)
trainval_df = pd.concat([train_df, val_df], ignore_index=True)

# Fit Y models with ROCV
rf_model,  rf_params,  rf_rocv_mae  = optimize_rf_rocv(trainval_df)
xgb_model, xgb_params, xgb_rocv_mae = optimize_xgb_rocv(trainval_df)

print("\n=== ROCV Best Params ===")
print("RF :", rf_params,  f"| ROCV_MAE={rf_rocv_mae:.2f}")
print("XGB:", xgb_params, f"| ROCV_MAE={xgb_rocv_mae:.2f}")

# ------------------ VAL evaluation helpers ------------------
def y_ensemble_weights(y_true, yhat_rf, yhat_xgb, eps=1e-6):
    mae_rf  = mean_absolute_error(y_true, yhat_rf)
    mae_xgb = mean_absolute_error(y_true, yhat_xgb)
    w_rf, w_xgb = 1.0/(mae_rf+eps), 1.0/(mae_xgb+eps)
    s = w_rf + w_xgb
    return w_rf/s, w_xgb/s, mae_rf, mae_xgb

def eval_exog_on_val(exog_val):
    """Return Y-ENS weights & metrics & residuals on VAL for a given exog table."""
    hist_for_val = train_df[["ds","y","orders","stock","month","year"]].copy()
    prf,_  = recursive_forward_predict_y(rf_model,  FEATURES_Y, hist_for_val.copy(), exog_val, VAL_START, VAL_END)
    pxgb,_ = recursive_forward_predict_y(xgb_model, FEATURES_Y, hist_for_val.copy(), exog_val, VAL_START, VAL_END)
    join = val_df[["ds","y"]].merge(prf, on="ds", how="left").rename(columns={"yhat":"yhat_rf"}) \
                             .merge(pxgb, on="ds", how="left").rename(columns={"yhat":"yhat_xgb"})
    w_rf, w_xgb, mae_rf, mae_xgb = y_ensemble_weights(join["y"].values, join["yhat_rf"].values, join["yhat_xgb"].values)
    yhat_ens = w_rf*join["yhat_rf"].values + w_xgb*join["yhat_xgb"].values
    mae_ens, rmse_ens, mape_ens = mae_rmse_mape(join["y"].values, yhat_ens)
    res = {
        "weights": (w_rf, w_xgb),
        "mae_rf": mae_rf, "mae_xgb": mae_xgb,
        "mae_ens": mae_ens, "rmse_ens": rmse_ens, "mape_ens": mape_ens,
        "residuals": {"RF": (join["y"].values - join["yhat_rf"].values),
                      "XGB": (join["y"].values - join["yhat_xgb"].values),
                      "ENS": (join["y"].values - yhat_ens)}
    }
    return res, join

# ------------------ Build all VAL exogs ------------------
def build_exog(method, df_all, start_ds, end_ds, cutoff,
               cache_val=None, cache_test_full=None, cache_test_short=None,
               val_reports=None):
    """Generic exog builder for single methods. Composite handled separately."""
    if method == "Prophet":
        return build_exog_univariate_for_period(df_all, start_ds, end_ds, cutoff, "prophet")
    if method == "SARIMA":
        return build_exog_univariate_for_period(df_all, start_ds, end_ds, cutoff, "sarima")
    if method == "ETS":
        return build_exog_univariate_for_period(df_all, start_ds, end_ds, cutoff, "ets")
    if method == "Ensemble":
        return build_exog_inverse_mae_for_period(df_all, start_ds, end_ds, cutoff)
    if method == "ML-Exog RF":
        return build_exog_ml_for_period(df_all, start_ds, end_ds, cutoff, "rf")
    if method == "ML-Exog XGB":
        return build_exog_ml_for_period(df_all, start_ds, end_ds, cutoff, "xgb")
    raise ValueError("Unknown basic exog method")

BASIC_METHODS = ["Prophet","SARIMA","ETS","Ensemble","ML-Exog RF","ML-Exog XGB"]

# Build & score VAL for basics
exog_val_tables = {}
val_method_reports = {}
for m in BASIC_METHODS:
    exog_val = build_exog(m, df_raw[["ds","orders","stock"]], VAL_START, VAL_END, cutoff=VAL_START)
    exog_val_tables[m] = exog_val
    rep, _ = eval_exog_on_val(exog_val)
    val_method_reports[m] = rep

# ------------------ Composite exogs on VAL: ML RF+XGB & All-Ensemble (5) & Top2/Top3 ------------------
def combine_exogs_weighted(exog_dict, weights):
    """exog_dict: {name: df_exog}; weights: {name: float}. All exogs share same ds index."""
    names = [k for k in weights.keys() if k in exog_dict]
    base = exog_dict[names[0]][["ds"]].copy()
    out = base.copy()
    for col in ["orders","stock"]:
        s = np.zeros(len(base), dtype=float)
        for nm in names:
            s += float(weights[nm]) * pd.to_numeric(exog_dict[nm][col].values, errors="coerce")
        out[col] = s
    return _postprocess_exog(out)

# ML RF+XGB weights by VAL Y-ENS MAE
ml_pair = ["ML-Exog RF","ML-Exog XGB"]
mae_pair = {m: val_method_reports[m]["mae_ens"] for m in ml_pair}
inv = {m: (1.0/mae_pair[m]) if np.isfinite(mae_pair[m]) and mae_pair[m]>0 else 0.0 for m in ml_pair}
s = sum(inv.values()) if sum(inv.values())>0 else 1.0
w_ml_ens = {m: inv[m]/s for m in ml_pair}

# All-Ensemble-5 across Prophet,SARIMA,ETS,MLRF,MLXGB (exclude 'Ensemble' to avoid double-counting)
all5 = ["Prophet","SARIMA","ETS","ML-Exog RF","ML-Exog XGB"]
mae_all5 = {m: val_method_reports[m]["mae_ens"] for m in all5}
inv_all5 = {m:(1.0/mae_all5[m]) if np.isfinite(mae_all5[m]) and mae_all5[m]>0 else 0.0 for m in all5}
s5 = sum(inv_all5.values()) if sum(inv_all5.values())>0 else 1.0
w_all5 = {m: inv_all5[m]/s5 for m in all5}

# Top2 / Top3 by VAL MAE among all5
ranked = sorted(all5, key=lambda m: (mae_all5[m], m))
top2 = ranked[:2]; top3 = ranked[:3]
inv_top2 = {m:(1.0/mae_all5[m]) if mae_all5[m]>0 else 0.0 for m in top2}
st2 = sum(inv_top2.values()) if sum(inv_top2.values())>0 else 1.0
w_top2 = {m: inv_top2[m]/st2 for m in top2}
inv_top3 = {m:(1.0/mae_all5[m]) if mae_all5[m]>0 else 0.0 for m in top3}
st3 = sum(inv_top3.values()) if sum(inv_top3.values())>0 else 1.0
w_top3 = {m: inv_top3[m]/st3 for m in top3}

# Build composite VAL tables
exog_val_tables["ML-Exog RF+XGB"] = combine_exogs_weighted({k: exog_val_tables[k] for k in ml_pair}, w_ml_ens)
rep_ml_ens, _ = eval_exog_on_val(exog_val_tables["ML-Exog RF+XGB"])
val_method_reports["ML-Exog RF+XGB"] = rep_ml_ens

exog_val_tables["All-Ensemble-5"] = combine_exogs_weighted({k: exog_val_tables[k] for k in all5}, w_all5)
rep_all5, _ = eval_exog_on_val(exog_val_tables["All-Ensemble-5"])
val_method_reports["All-Ensemble-5"] = rep_all5

exog_val_tables["All-Ensemble-Top2"] = combine_exogs_weighted({k: exog_val_tables[k] for k in top2}, w_top2)
rep_top2, _ = eval_exog_on_val(exog_val_tables["All-Ensemble-Top2"])
val_method_reports["All-Ensemble-Top2"] = rep_top2

exog_val_tables["All-Ensemble-Top3"] = combine_exogs_weighted({k: exog_val_tables[k] for k in top3}, w_top3)
rep_top3, _ = eval_exog_on_val(exog_val_tables["All-Ensemble-Top3"])
val_method_reports["All-Ensemble-Top3"] = rep_top3

ALL_METHODS = BASIC_METHODS + ["ML-Exog RF+XGB","All-Ensemble-5","All-Ensemble-Top2","All-Ensemble-Top3"]

# ------------------ VAL summary & print ------------------
rows = []
for m in ALL_METHODS:
    rep = val_method_reports[m]
    w_rf, w_xgb = rep["weights"]
    rows.append([m, rep["mae_rf"], rep["mae_xgb"], rep["mae_ens"], rep["rmse_ens"], rep["mape_ens"], w_rf, w_xgb])
val_sel_table = pd.DataFrame(rows, columns=["Exog","VAL_MAE_RF","VAL_MAE_XGB","VAL_MAE_YENS","VAL_RMSE_YENS","VAL_MAPE_YENS","w_RF","w_XGB"]) \
                  .sort_values("VAL_MAE_YENS")
val_sel_table.to_csv("outputs/val_exog_selection_full.csv", index=False)

print("\n=== VAL Exog Selection (all methods) ===")
print(val_sel_table.to_string(index=False))

best_exog = val_sel_table.iloc[0]["Exog"]
print(f"\nSelected Exog by Y-ENS VAL MAE: {best_exog}")

# ------------------ Build TEST exog tables for ALL methods ------------------
def build_test_exog_for_method(method):
    df_all = df_raw[["ds","orders","stock"]]
    if method in BASIC_METHODS:
        full  = build_exog(method, df_all, TEST_START, TEST_END,        cutoff=TEST_START)
        short = build_exog(method, df_all, TEST_START, TEST_END_SHORT,  cutoff=TEST_START)
        return full, short
    if method == "ML-Exog RF+XGB":
        ex_rf_full,  ex_rf_short  = build_test_exog_for_method("ML-Exog RF")
        ex_xgb_full, ex_xgb_short = build_test_exog_for_method("ML-Exog XGB")
        # weights from VAL
        weights = w_ml_ens
        full  = combine_exogs_weighted({"ML-Exog RF":ex_rf_full, "ML-Exog XGB":ex_xgb_full}, weights)
        short = combine_exogs_weighted({"ML-Exog RF":ex_rf_short,"ML-Exog XGB":ex_xgb_short}, weights)
        return full, short
    if method in ["All-Ensemble-5","All-Ensemble-Top2","All-Ensemble-Top3"]:
        # Build base pieces
        parts = {}
        for m in all5:
            f,s = build_test_exog_for_method(m)
            parts[m] = (f,s)
        weights = w_all5 if method=="All-Ensemble-5" else (w_top2 if method=="All-Ensemble-Top2" else w_top3)
        full  = combine_exogs_weighted({m:parts[m][0] for m in weights.keys()}, weights)
        short = combine_exogs_weighted({m:parts[m][1] for m in weights.keys()}, weights)
        return full, short
    raise ValueError("Unknown exog method in test builder")

exog_test_full  = {}
exog_test_short = {}
for m in ALL_METHODS:
    f,s = build_test_exog_for_method(m)
    exog_test_full[m]  = f
    exog_test_short[m] = s

# ------------------ Predict & evaluate on TEST (for ALL exogs × Y variants) ------------------
def add_bootstrap_intervals(pred_df, residuals, B=B_BOOT):
    if residuals.size == 0:
        scale = max(1e-6, np.median(np.abs(pred_df["yhat"].values)) * 0.1)
        sims = pred_df["yhat"].to_numpy().reshape(-1,1) + rng_boot.laplace(0.0, scale, size=(len(pred_df), B))
    else:
        idx = rng_boot.integers(0, residuals.size, size=(len(pred_df), B))
        noise = residuals[idx]; sims = pred_df["yhat"].to_numpy().reshape(-1,1) + noise
    lo80 = np.nanquantile(sims, 0.10, axis=1)
    hi80 = np.nanquantile(sims, 0.90, axis=1)
    lo95 = np.nanquantile(sims, 0.025, axis=1)
    hi95 = np.nanquantile(sims, 0.975, axis=1)
    out = pred_df.copy()
    out["pi80_lo"] = lo80; out["pi80_hi"] = hi80
    out["pi95_lo"] = lo95; out["pi95_hi"] = hi95
    return out, sims

def infer_starting_stock(df_raw, test_start, override=None):
    if override is not None: return float(override)
    prev = df_raw[df_raw["ds"] < test_start].tail(1)
    if "stock" in prev.columns and len(prev):
        return float(max(0.0, prev["stock"].iloc[0]))
    return 0.0

def stockout_probability(starting_stock, sims_matrix):
    sims = np.maximum(0.0, sims_matrix)
    cum = np.cumsum(sims, axis=0)
    tts = np.full(sims.shape[1], np.nan)
    for b in range(sims.shape[1]):
        idx = np.where(cum[:,b] >= starting_stock)[0]
        if idx.size>0: tts[b] = idx[0] + 1
    p3 = np.mean(np.nan_to_num(tts, nan=np.inf) <= 3)
    p6 = np.mean(np.nan_to_num(tts, nan=np.inf) <= 6)
    exp_t = np.nanmean(tts) if np.any(~np.isnan(tts)) else np.nan
    return float(p3), float(p6), (None if np.isnan(exp_t) else float(exp_t))

hist_min = trainval_df[["ds","y","orders","stock","month","year"]].copy()

# Use method-specific Y-ENS weights from VAL when variant==Y-ENS
def predict_with_variant(exog_table, variant, val_weights_for_method):
    if variant == "RF":
        p,_ = recursive_forward_predict_y(rf_model,  FEATURES_Y, hist_min.copy(), exog_table, TEST_START, exog_table["ds"].max())
        resids = None  # will be set by VAL residuals of RF for the exog method
    elif variant == "XGB":
        p,_ = recursive_forward_predict_y(xgb_model, FEATURES_Y, hist_min.copy(), exog_table, TEST_START, exog_table["ds"].max())
        resids = None
    else:  # Y-ENS
        prf,_  = recursive_forward_predict_y(rf_model,  FEATURES_Y, hist_min.copy(), exog_table, TEST_START, exog_table["ds"].max())
        pxgb,_ = recursive_forward_predict_y(xgb_model, FEATURES_Y, hist_min.copy(), exog_table, TEST_START, exog_table["ds"].max())
        p = prf.merge(pxgb, on="ds", suffixes=("_rf","_xgb"))
        w_rf, w_xgb = val_weights_for_method
        p["yhat"] = w_rf*p["yhat_rf"] + w_xgb*p["yhat_xgb"]
        p = p[["ds","yhat"]]
    return p

# Prepare VAL residuals per method for each variant
# (RF/XGB residuals are fixed per method; Y-ENS residuals use that method's Y-ENS weights)
val_residuals_by_method = {}
for m in ALL_METHODS:
    rep = val_method_reports[m]
    val_residuals_by_method[m] = {
        "RF":  np.array(rep["residuals"]["RF"], dtype=float),
        "XGB": np.array(rep["residuals"]["XGB"], dtype=float),
        "Y-ENS": np.array(rep["residuals"]["ENS"], dtype=float),
        "weights": rep["weights"]
    }

VARIANTS = ["RF","XGB","Y-ENS"]
results_rows = []

for horizon, ds_end, exog_pool in [("Full", TEST_END, exog_test_full),
                                   ("Short3", TEST_END_SHORT, exog_test_short)]:
    for m in ALL_METHODS:
        exog_tbl = exog_pool[m]
        for var in VARIANTS:
            preds = predict_with_variant(exog_tbl, var, val_residuals_by_method[m]["weights"])
            # PI using VAL residuals of SAME variant
            resids = val_residuals_by_method[m][var]
            preds_pi, sims = add_bootstrap_intervals(preds, resids, B=B_BOOT)

            # Evaluate if truth exists
            truth = df_raw[(df_raw["ds"] >= TEST_START) & (df_raw["ds"] <= ds_end)][["ds","y"]].copy()
            eval_df = truth.merge(preds_pi, on="ds", how="left")
            mae, rmse, mape = mae_rmse_mape(eval_df["y"], eval_df["yhat"])

            # Stockout
            starting_stock = infer_starting_stock(df_raw, TEST_START, STARTING_STOCK_OVERRIDE)
            p3, p6, exp_t = stockout_probability(starting_stock, sims)

            # Save per-combo CSV
            out_path = f"outputs/preds_{horizon}_{m}_{var.replace('-','')}.csv".replace(' ','_')
            eval_df.to_csv(out_path, index=False)

            # Row
            wrf, wxgb = val_residuals_by_method[m]["weights"]
            results_rows.append([horizon, m, var, mae, rmse, mape, wrf if var=="Y-ENS" else np.nan,
                                 wxgb if var=="Y-ENS" else np.nan, p3, p6, exp_t])

# Summaries
summary_cols = ["Horizon","Exog","Y-Variant","MAE","RMSE","MAPE","w_RF","w_XGB","P_stockout_3m","P_stockout_6m","E_T_stockout_mo"]
summary_all = pd.DataFrame(results_rows, columns=summary_cols) \
                .sort_values(["Horizon","Exog","Y-Variant"])
summary_all.to_csv("outputs/test_summary_all_exogs.csv", index=False)

print("\n=== TEST Summary — ALL Exogs × Y Variants ===")
print(summary_all.to_string(index=False))

# Ayrıca "seçilen en iyi exog" için kısa özet:
sel_mask = (summary_all["Exog"] == best_exog)
print(f"\n=== TEST Summary — SELECTED EXOG: {best_exog} ===")
print(summary_all[sel_mask].to_string(index=False))

# ------------------ SHAP / Feature Importances ------------------
def dump_feature_importances(model, model_name, X_df):
    out = pd.DataFrame({"feature": X_df.columns, "importance": getattr(model, "feature_importances_", np.zeros(X_df.shape[1]))})
    out = out.sort_values("importance", ascending=False)
    out.to_csv(f"outputs/feat_importance_{model_name}.csv", index=False)
    return out

def dump_shap_values(model, model_name, X_df):
    if not HAVE_SHAP:
        return None
    try:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_df)
        sv_arr = np.array(sv)
        if sv_arr.ndim == 3:  # (classes, n, f)
            sv_arr = np.mean(np.abs(sv_arr), axis=0)
        mean_abs = np.mean(np.abs(sv_arr), axis=0)
        out = pd.DataFrame({"feature": X_df.columns, "mean_abs_shap": mean_abs}).sort_values("mean_abs_shap", ascending=False)
        out.to_csv(f"outputs/shap_importance_{model_name}.csv", index=False)
        return out
    except Exception:
        return None

X_trainval = trainval_df[FEATURES_Y].copy()
fi_rf  = dump_feature_importances(rf_model,  "rf",  X_trainval)
fi_xgb = dump_feature_importances(xgb_model, "xgb", X_trainval)
shap_rf  = dump_shap_values(rf_model,  "rf",  X_trainval)
shap_xgb = dump_shap_values(xgb_model, "xgb", X_trainval)

# ------------------ Simple Plots for Selected Exog ------------------
def plot_with_pi(eval_df, title, savepath):
    plt.figure(figsize=(12,6))
    plt.plot(eval_df["ds"], eval_df["y"], "o-", label="Gerçek")
    plt.plot(eval_df["ds"], eval_df["yhat"], "--", label="Tahmin")
    if {"pi80_lo","pi80_hi"}.issubset(eval_df.columns):
        plt.fill_between(eval_df["ds"], eval_df["pi80_lo"], eval_df["pi80_hi"], alpha=0.25, label="PI 80%")
    if {"pi95_lo","pi95_hi"}.issubset(eval_df.columns):
        plt.fill_between(eval_df["ds"], eval_df["pi95_lo"], eval_df["pi95_hi"], alpha=0.15, label="PI 95%")
    plt.title(title); plt.xlabel("Tarih"); plt.ylabel("Satış"); plt.legend(); plt.tight_layout()
    plt.savefig(savepath, dpi=160); plt.close()

# Selected exog plots (Full) for each Y-variant
for var in VARIANTS:
    path_csv = f"outputs/preds_Full_{best_exog}_{var.replace('-','')}.csv".replace(' ','_')
    if os.path.exists(path_csv):
        dfp = pd.read_csv(path_csv, parse_dates=["ds"])
        plot_with_pi(dfp, f"{best_exog} • {var} • Full", f"outputs/plot_full_{best_exog}_{var.replace('-','')}.png".replace(' ','_'))

print("\nArtifacts:")
print("- outputs/val_exog_selection_full.csv (TÜM exogların VAL kıyası + Y-ENS ağırlıkları)")
print("- outputs/test_summary_all_exogs.csv   (TEST sonuçları: tüm exog × Y-variant)")
print("- outputs/preds_<Horizon>_<Exog>_<Variant>.csv (tahmin+PI)")
print("- outputs/plot_full_<SelectedExog>_<Variant>.png")
print("- outputs/feat_importance_*.csv / shap_importance_*.csv")


KeyboardInterrupt: 